# POS Cash Balance – Feature Engineering

Builds customer-level features from the `POS_CASH_balance` dataset for use in the Home Credit default prediction pipeline.

**Pipeline overview**
1. Load raw data
2. Impute missing installment counts
3. Monthly aggregation per customer
4. DPD feature engineering (all-time, 1-year, 3-month windows)
5. Prepayment flag derivation
6. Last-recorded-month features
7. Previous-loan enrichment
8. Merge all features → final output

## 1. Imports

In [2]:
import os
import pandas as pd
import numpy as np

## 2. Configuration & Paths

In [3]:
# ── Project data root ──────────────────────────────────────────────────────────
DATA_ROOT = os.environ.get("HOMECREDIT_DATA_ROOT", r"a:\Dominic\My Projects\Home Credit\data")

# ── Input paths ───────────────────────────────────────────────────────────
PATH_POS_BAL       = os.path.join(DATA_ROOT, "Input", "POS_CASH_balance.csv")
PATH_APP_TRAIN     = os.path.join(DATA_ROOT, "Input", "application_train.csv")
PATH_PREV_LOANS    = os.path.join(DATA_ROOT, "Staging",
                                   "prev_loan_data_processed.parquet")

# ── Output path ───────────────────────────────────────────────────────────
PATH_OUTPUT        = os.path.join(DATA_ROOT, "Staging",
                                   "posdata_processed.parquet")

## 3. Load Data

In [4]:
# Previous loan metadata (used for enrichment in Section 7)
PrevLoans = pd.read_parquet(PATH_PREV_LOANS)
PrevLoans = PrevLoans.astype({'SK_ID_PREV': 'string', 'SK_ID_CURR': 'string'})

# POS Cash balance – sort chronologically within each loan
POSBal = pd.read_csv(PATH_POS_BAL)
POSBal = POSBal.astype({'SK_ID_PREV': 'string', 'SK_ID_CURR': 'string'})
POSBal.sort_values(
    by=['SK_ID_CURR', 'SK_ID_PREV', 'MONTHS_BALANCE'],
    inplace=True,
    ignore_index=True,
)

# Application base – used to filter to training customers and append TARGET
base = pd.read_csv(PATH_APP_TRAIN, usecols=['SK_ID_CURR', 'TARGET'])
base = base.astype({'SK_ID_CURR': 'string'})

# Restrict POS data to customers present in the training set
POSBal = base.merge(POSBal, on='SK_ID_CURR', how='inner').drop(columns=['TARGET'])

print(f"POSBal shape : {POSBal.shape}")
print(f"PrevLoans shape: {PrevLoans.shape}")

POSBal shape : (8543375, 8)
PrevLoans shape: (1670214, 40)


## 4. Impute Missing Installment Counts

Forward-fill then back-fill (limit = 3 months) within each loan (`SK_ID_PREV`).
A limit of 3 keeps imputation local; isolated gaps larger than 3 remain `NaN`.

In [4]:
for col in ['CNT_INSTALMENT', 'CNT_INSTALMENT_FUTURE']:
    POSBal[col] = (
        POSBal
        .groupby('SK_ID_PREV')[col]
        .transform(lambda x: x.ffill(limit=3).bfill(limit=3))
    )

print("Remaining nulls after imputation:")
print(POSBal[['CNT_INSTALMENT', 'CNT_INSTALMENT_FUTURE']].isna().sum())

Remaining nulls after imputation:
CNT_INSTALMENT           1606
CNT_INSTALMENT_FUTURE    1633
dtype: int64


**Note**: Check for prepayments

## 5. Aggregate to Monthly Customer Level

When a customer holds multiple simultaneous POS contracts, collapse them into
a single row per `(SK_ID_CURR, MONTHS_BALANCE)` by taking the **max** of each
metric — this captures the worst-case DPD exposure for that month.

In [ ]:
POSBal_Mon = (
    POSBal
    .groupby(['SK_ID_CURR', 'MONTHS_BALANCE'])[
        ['CNT_INSTALMENT', 'CNT_INSTALMENT_FUTURE', 'SK_DPD', 'SK_DPD_DEF']
    ]
    .agg('max')
    .reset_index()
)

# Rename columns to upper-case (already flat after scalar agg)
POSBal_Mon.columns = POSBal_Mon.columns.str.upper()

print(f"POSBal_Mon shape: {POSBal_Mon.shape}")
POSBal_Mon.head(3)

POSBal_Mon shape: (7350685, 6)


,SK_ID_CURR,MONTHS_BALANCE,CNT_INSTALMENT,CNT_INSTALMENT_FUTURE,SK_DPD,SK_DPD_DEF
0,100002,-19,24.0,24.0,0,0
1,100002,-18,24.0,23.0,0,0
2,100002,-17,24.0,22.0,0,0


## 6. DPD Feature Engineering

Three time windows are computed independently and merged later:

| Suffix | Window |
|--------|--------|
| *(none)* | All available history |
| `_3Y` | Last 36 months (`MONTHS_BALANCE >= -36`) |
| `_2Y` | Last 12 months (`MONTHS_BALANCE >= -24`) |
| `_1Y` | Last 12 months (`MONTHS_BALANCE >= -12`) |
| `_6M` | Last 6 months (`MONTHS_BALANCE >= -6`) |
| `_3M` | Last 3 months (`MONTHS_BALANCE >= -3`) |

<br>

- `MONTHS_BALANCE` is negative (0 = most recent month). 
- Filtering `>= -12` selects the 12 most recent months.

In [ ]:
def build_dpd_features(df, suffix=''):
    """Aggregate DPD metrics at SK_ID_CURR level.

    Parameters
    ----------
    df : DataFrame with columns SK_ID_CURR, SK_DPD, SK_DPD_DEF
    suffix : str – appended to each output column name
    """
    return df.groupby('SK_ID_CURR').agg(
        **{
            f'POS_DAYS_IN_DPD{suffix}': ('SK_DPD', lambda x: (x > 0).sum()),
            f'POS_DAYS_IN_DPD_DEF{suffix}': ('SK_DPD_DEF', lambda x: (x > 0).sum()),
            f'POS_DPD_TO_DURATION_RATIO{suffix}': ('SK_DPD', lambda x: (x > 0).sum() / len(x) if len(x) > 0 else 0),
        }
    )

pos_aggregated     = build_dpd_features(POSBal_Mon)
pos_aggregated_3Y  = build_dpd_features(
    POSBal_Mon[POSBal_Mon['MONTHS_BALANCE'] >= -36], suffix='_3Y'
)
pos_aggregated_2Y  = build_dpd_features(
    POSBal_Mon[POSBal_Mon['MONTHS_BALANCE'] >= -24], suffix='_2Y'
)
pos_aggregated_1Y  = build_dpd_features(
    POSBal_Mon[POSBal_Mon['MONTHS_BALANCE'] >= -12], suffix='_1Y'
)
pos_aggregated_6M  = build_dpd_features(
    POSBal_Mon[POSBal_Mon['MONTHS_BALANCE'] >= -6], suffix='_6M'
)
pos_aggregated_3M  = build_dpd_features(
    POSBal_Mon[POSBal_Mon['MONTHS_BALANCE'] >= -3], suffix='_3M'
)

print("All-time :", pos_aggregated.shape)
print("Last 36 M:", pos_aggregated_3Y.shape)
print("Last 24 M:", pos_aggregated_2Y.shape)
print("Last 12 M:", pos_aggregated_1Y.shape)
print("Last 6 M :", pos_aggregated_6M.shape)
print("Last 3 M :", pos_aggregated_3M.shape)

All-time : (289444, 3)
Last 12 M: (206897, 3)
Last 3 M : (139631, 3)


## 7. Prepayment Flag

A loan is flagged as **prepaid** when its originally scheduled instalment count
(`CNT_INSTALMENT` at the *first* observed month) differs from the count at the
*last* observed month — signalling that the repayment schedule was shortened.

`PREV_CREDIT_PREPAYMENT_FLAG = 1` if the instalment count changed, else `0`.

In [10]:
# First and last observed record per loan
POSBal_first = POSBal.groupby(['SK_ID_CURR', 'SK_ID_PREV']).first().reset_index()
POSBal_last  = POSBal.groupby(['SK_ID_CURR', 'SK_ID_PREV']).last().reset_index()

# Merge first vs last instalment counts on the same loan
POSBal_CNT_INSTALMENT = (
    POSBal_first[['SK_ID_CURR', 'SK_ID_PREV', 'CNT_INSTALMENT']]
    .merge(
        POSBal_last[['SK_ID_CURR', 'SK_ID_PREV', 'CNT_INSTALMENT']],
        on=['SK_ID_CURR', 'SK_ID_PREV'],
        suffixes=('_FIRST', '_LAST'),
    )
)
POSBal_CNT_INSTALMENT['PREV_CREDIT_PREPAYMENT_FLAG'] = (
    POSBal_CNT_INSTALMENT['CNT_INSTALMENT_FIRST']
    != POSBal_CNT_INSTALMENT['CNT_INSTALMENT_LAST']
).astype(int)

# Customer-level flag: 1 if ANY of their loans was prepaid
POSBal_Prepay_flag = (
    POSBal_CNT_INSTALMENT
    .groupby('SK_ID_CURR')['PREV_CREDIT_PREPAYMENT_FLAG']
    .max()
    .reset_index()
)

print("Prepayment rate:",
      f"{POSBal_Prepay_flag['PREV_CREDIT_PREPAYMENT_FLAG'].mean():.2%}")

Prepayment rate: 52.27%


## 8. Last-Recorded-Month Features

Aggregates based on the most recent month observed per loan (`POSBal_last`),
then summarised at customer level.

In [11]:
POSBal_Agg_last = POSBal_last.groupby('SK_ID_CURR').agg(
    POS_AVG_CNT_INSTALMENT_LAST_MONTH=('CNT_INSTALMENT', 'mean'),
    POS_AVG_CNT_INSTALMENT_FUTURE_LAST_MONTH=('CNT_INSTALMENT_FUTURE', 'mean'),
    POS_MAX_CNT_INSTALMENT_LAST_MONTH=('CNT_INSTALMENT', 'max'),
    POS_MAX_CNT_INSTALMENT_FUTURE_LAST_MONTH=('CNT_INSTALMENT_FUTURE', 'max'),
    POS_CNT_ACTIVE_PREV_CONTRACT_LAST_MONTH=('NAME_CONTRACT_STATUS', lambda x: (x == 'Active').sum()),
    POS_SK_DPD_LAST_MONTH=('SK_DPD', 'max'),
    POS_SK_DPD_DEF_LAST_MONTH=('SK_DPD_DEF', 'max'),
)

print(f"Last-month features shape: {POSBal_Agg_last.shape}")

Last-month features shape: (289444, 7)


## 9. Months Since Last DPD Event

Filters to month where `SK_DPD > 0` first, then takes the most recent such month.

Result: `POS_MONTHS_SINCE_LAST_DPD_GT0` = number of months ago the customer
last had a DPD > 0 on any POS loan. Customers with no DPD history get `NaN`.

In [ ]:
# ── FIXED: filter to DPD > 0 months before taking the max ───────────────
POSBal_latest_dpd = (
    POSBal[POSBal['SK_DPD'] > 0]          # only months with a DPD event
    .groupby('SK_ID_CURR')['MONTHS_BALANCE']
    .max()                                  # most recent DPD month (least negative)
    .reset_index()
    .rename(columns={'MONTHS_BALANCE': 'POS_MONTHS_SINCE_LAST_DPD_GT0'})
)
# MONTHS_BALANCE is negative; negate to get 'months ago'
POSBal_latest_dpd['POS_MONTHS_SINCE_LAST_DPD_GT0'] = (-POSBal_latest_dpd['POS_MONTHS_SINCE_LAST_DPD_GT0'])

print(f"Customers with at least one DPD event: {len(POSBal_latest_dpd):,}")
POSBal_latest_dpd['POS_MONTHS_SINCE_LAST_DPD_GT0'].describe()

Customers with at least one DPD event: 54,683


count    54683.000000
mean        42.139696
std         26.902085
min          1.000000
25%         18.000000
50%         41.000000
75%         65.000000
max         96.000000
Name: POS_MONTHS_SINCE_LAST_DPD_GT0, dtype: float64

## 10. Previous-Loan Enrichment

Join the prepayment table with `PrevLoans` metadata to compute
loan-level ratios, then aggregate to customer level.

In [ ]:
POSBal_PrevLoans = POSBal_CNT_INSTALMENT.merge(
    PrevLoans,
    on=['SK_ID_PREV', 'SK_ID_CURR'],
    how='left',
)

POSBal_Agg_PrevLoans = POSBal_PrevLoans.groupby('SK_ID_CURR').agg(
    POS_TOTAL_PREV_LOAN=('SK_ID_PREV', 'nunique'),
    POS_SUM_PREV_CREDIT_PREPAYMENT_FLAG=('PREV_CREDIT_PREPAYMENT_FLAG', 'sum'),
    POS_AVG_CREDIT_TO_APPL=('CREDIT_TO_APPL', 'mean'),
    POS_ANNUITY_TO_CREDIT_RATIO=('AMT_ANNUITY_TO_CREDIT_RATIO', 'mean'),
    POS_CNT_HIGH_YIELD_GROUP=('NAME_YIELD_GROUP', lambda x: (x == 'high').sum()),
    POS_CNT_MIDDLE_YIELD_GROUP=('NAME_YIELD_GROUP', lambda x: (x == 'middle').sum()),
    POS_CNT_INSURED_ON_APPROVAL=('NFLAG_INSURED_ON_APPROVAL', 'sum'),
    POS_AVG_DAYS_DECISION=('DAYS_DECISION', 'mean'),
)

print(f"PrevLoan enrichment shape: {POSBal_Agg_PrevLoans.shape}")

PrevLoan enrichment shape: (289444, 8)


## 11. Free Intermediate Memory

In [14]:
del PrevLoans, POSBal_Mon, POSBal_first, POSBal_last
del POSBal_CNT_INSTALMENT, POSBal_PrevLoans
import gc; gc.collect()
print("Intermediate DataFrames released.")

Intermediate DataFrames released.


## 12. Merge All Features

Build the final customer-level feature table by left-joining all components
onto the all-time DPD base. Indicator flags (`POS_IS_IN_LAST_1Y`,
`POS_IS_IN_LAST_3M`) record whether a customer had *any* activity in those windows.

In [15]:
def flag_merge(left, right, on, flag_col):
    """Left-merge and add a binary flag indicating matched rows."""
    merged = left.merge(right, on=on, how='left', indicator=True)
    merged[flag_col] = (merged['_merge'] == 'both').astype(int)
    return merged.drop(columns=['_merge'])


final_pos_aggregated = pos_aggregated.copy()

# DPD windows
final_pos_aggregated = flag_merge(
    final_pos_aggregated, pos_aggregated_3Y, 'SK_ID_CURR', 'POS_IS_IN_LAST_3Y'
)
final_pos_aggregated = flag_merge(
    final_pos_aggregated, pos_aggregated_2Y, 'SK_ID_CURR', 'POS_IS_IN_LAST_2Y'
)
final_pos_aggregated = flag_merge(
    final_pos_aggregated, pos_aggregated_1Y, 'SK_ID_CURR', 'POS_IS_IN_LAST_1Y'
)
final_pos_aggregated = flag_merge(
    final_pos_aggregated, pos_aggregated_6M, 'SK_ID_CURR', 'POS_IS_IN_LAST_6M'
)
final_pos_aggregated = flag_merge(
    final_pos_aggregated, pos_aggregated_3M, 'SK_ID_CURR', 'POS_IS_IN_LAST_3M'
)

# Last-month contract / instalment features
final_pos_aggregated = final_pos_aggregated.merge(
    POSBal_Agg_last, on='SK_ID_CURR', how='left'
)

# Months since last DPD (NaN for customers with no DPD history)
final_pos_aggregated = final_pos_aggregated.merge(
    POSBal_latest_dpd, on='SK_ID_CURR', how='left'
)

# Prepayment flag (customer-level)
final_pos_aggregated = final_pos_aggregated.merge(
    POSBal_Prepay_flag, on='SK_ID_CURR', how='left'
)

# Previous-loan enrichment features
final_pos_aggregated = final_pos_aggregated.merge(
    POSBal_Agg_PrevLoans, on='SK_ID_CURR', how='left'
)

print(f"Final shape: {final_pos_aggregated.shape}")
final_pos_aggregated.head(3)

Final shape: (289444, 41)


,SK_ID_CURR,POS_DAYS_IN_DPD,POS_DAYS_IN_DPD_DEF,POS_DPD_TO_DURATION_RATIO,POS_DAYS_IN_DPD_3Y,POS_DAYS_IN_DPD_DEF_3Y,POS_DPD_TO_DURATION_RATIO_3Y,POS_IS_IN_LAST_3Y,POS_DAYS_IN_DPD_2Y,POS_DAYS_IN_DPD_DEF_2Y,...,POS_MONTHS_SINCE_LAST_DPD_GT0,PREV_CREDIT_PREPAYMENT_FLAG,POS_TOTAL_PREV_LOAN,POS_SUM_PREV_CREDIT_PREPAYMENT_FLAG,POS_AVG_CREDIT_TO_APPL,POS_ANNUITY_TO_CREDIT_RATIO,POS_CNT_HIGH_YIELD_GROUP,POS_CNT_MIDDLE_YIELD_GROUP,POS_CNT_INSURED_ON_APPROVAL,POS_AVG_DAYS_DECISION
0,100002,0,0,0.0,0.0,0.0,0.0,1,0.0,0.0,...,NaN,0,1,0,1.000000,0.051670,0,0,0.0,-606.0
1,100003,0,0,0.0,0.0,0.0,0.0,1,0.0,0.0,...,NaN,1,3,1,1.057664,0.126383,0,2,2.0,-1305.0
2,100004,0,0,0.0,0.0,0.0,0.0,1,0.0,0.0,...,NaN,1,1,1,0.828021,0.266450,0,1,0.0,-815.0


## 13. Quality Checks

Quick sanity checks before writing output.

In [16]:
# Null rates per feature
null_rates = final_pos_aggregated.isna().mean().sort_values(ascending=False)
print("Null rate (top 15 columns):")
print(null_rates.head(15).to_string())

# Duplicate customer IDs
n_dupes = final_pos_aggregated['SK_ID_CURR'].duplicated().sum()
assert n_dupes == 0, f"Duplicate SK_ID_CURR rows found: {n_dupes}"
print(f"\nNo duplicate SK_ID_CURR rows. Final row count: {len(final_pos_aggregated):,}")

Null rate (top 15 columns):
POS_MONTHS_SINCE_LAST_DPD_GT0    0.811076
POS_DPD_TO_DURATION_RATIO_3M     0.517589
POS_DAYS_IN_DPD_3M               0.517589
POS_DAYS_IN_DPD_DEF_3M           0.517589
POS_DAYS_IN_DPD_DEF_6M           0.426176
POS_DPD_TO_DURATION_RATIO_6M     0.426176
POS_DAYS_IN_DPD_6M               0.426176
POS_DAYS_IN_DPD_DEF_1Y           0.285192
POS_DPD_TO_DURATION_RATIO_1Y     0.285192
POS_DAYS_IN_DPD_1Y               0.285192
POS_DAYS_IN_DPD_2Y               0.152050
POS_DPD_TO_DURATION_RATIO_2Y     0.152050
POS_DAYS_IN_DPD_DEF_2Y           0.152050
POS_DAYS_IN_DPD_3Y               0.103958
POS_DPD_TO_DURATION_RATIO_3Y     0.103958

No duplicate SK_ID_CURR rows. Final row count: 289,444


## 14. Save Output

In [17]:
final_pos_aggregated.to_parquet(PATH_OUTPUT, index=False)
print(f"Saved → {PATH_OUTPUT}")

Saved → a:\Dominic\My Projects\Home Credit\src\intermediate\posdata_processed.parquet


## 15. Final Cleanup

In [ ]:
del (
    pos_aggregated, pos_aggregated_1Y, pos_aggregated_2Y, pos_aggregated_3Y, pos_aggregated_6M, pos_aggregated_3M,
    POSBal_Agg_last, POSBal_latest_dpd, POSBal_Prepay_flag,
    POSBal_Agg_PrevLoans, base, POSBal,
)
import gc; gc.collect()
print("All DataFrames released.")

All DataFrames released.
